# Data preparation: sources, cleaning and construction of the output gap

This notebook documents how the monthly dataset used in the study was assembled. It covers:

1. **Data sources and preparation**: where each series comes from and how it was harmonised.
2. **Screening and cleaning**: the outlier screen of the compiled panel and the one correction applied.
3. **Construction of the output gap**: splicing the three IIP base-year series and applying the Hodrick–Prescott filter, with checks against CEIC's linked series and against the `Output_Gap` column used in the analysis.

**Files needed**

| File | Contents | Where it goes |
|---|---|---|
| `IPI_94.xlsx`, `IPI_5.xlsx`, `IPI_11.xlsx` | CEIC exports of the IIP on the 1993-94, 2004-05 and 2011-12 bases | `data/IIP/` |
| `Compiled.xlsx` | Compiled monthly panel (sheet `X_var`) | `data/` |

**Running on Google Colab.** Run the first code cell. If the files are not already in `data/`, it opens an upload dialog: select all four files at once and they are moved into the right folders automatically. To use Google Drive instead, set `DRIVE_FOLDER` in that cell to the Drive folder that contains `data/`.

*The CEIC files are proprietary and are not distributed with the project's public repository.*

In [ ]:
# ---------------------------------------------------------------- setup (works on Colab and locally)
import os, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
warnings.filterwarnings("ignore", message="Could not infer format")

DRIVE_FOLDER = None            # e.g. "/content/drive/MyDrive/tpu-gpr-nse-volatility" to read files from Google Drive

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and DRIVE_FOLDER:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path(DRIVE_FOLDER)
else:
    BASE = Path(".").resolve()

IIP_DIR = BASE / "data" / "IIP"
IIP_DIR.mkdir(parents=True, exist_ok=True)
NEEDED = {"IPI_94.xlsx": IIP_DIR, "IPI_5.xlsx": IIP_DIR, "IPI_11.xlsx": IIP_DIR, "Compiled.xlsx": BASE / "data"}

missing = [f for f, d in NEEDED.items() if not (d / f).exists()]
if missing and IN_COLAB and not DRIVE_FOLDER:
    from google.colab import files
    print("Upload these files:", ", ".join(missing))
    uploaded = files.upload()
    for name in uploaded:
        clean = name.split(" (")[0] if " (" in name else name      # Colab renames duplicates "file (1).xlsx"
        if clean in NEEDED:
            shutil.move(name, NEEDED[clean] / clean)
    missing = [f for f, d in NEEDED.items() if not (d / f).exists()]

print("Base folder:", BASE)
print("Missing files:", missing if missing else "none")

FIG_DIR = BASE / "outputs" / "figures"; TAB_DIR = BASE / "outputs" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True); TAB_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------- plotting style and helpers
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100",
     "ink": "#0b0b0b", "ink2": "#52514e", "muted": "#898781", "grid": "#e1e0d9", "axis": "#c3c2b7"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10, "axes.titlesize": 11,
                     "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": C["grid"], "grid.linewidth": 0.6, "legend.frameon": False})

def save_table(df, name, index=True, show=True):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index)
    if show: display(df)
    return df

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show(); plt.close(fig)

def year_axis(ax, step=4):
    ax.xaxis.set_major_locator(mdates.YearLocator(step)); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

## 1. Data sources and preparation

The study uses monthly data from April 1999 to December 2025. Indian macroeconomic series were downloaded from **CEIC**; the uncertainty indices and the US volatility index were taken from the sites of the institutions that publish them; equity prices are for the NIFTY 50 index of the National Stock Exchange of India.

| Variable | Definition | Source | Raw frequency | Treatment |
|---|---|---|---|---|
| NIFTY 50 | Index closing level, end of month | National Stock Exchange of India | Monthly | Log returns; 12-month rolling volatility |
| IIP | Index of Industrial Production, 1993-94, 2004-05 and 2011-12 bases (MOSPI) | CEIC | Monthly | Ratio-spliced to 2011-12 = 100; HP-filtered → `Output_Gap` (Appendix A) |
| CPI_GR | Consumer price inflation, % year on year | CEIC | Monthly | As published |
| FX_USD | Indian rupee per US dollar, monthly average (RBI reference rate) | CEIC | Monthly | As published |
| Int_Rate | Weighted average call money rate, % | CEIC (RBI) | Monthly | As published; Mar-2007 replaced by RBI *Handbook of Statistics* value (14.07%) |
| TPU | Trade Policy Uncertainty index (Caldara et al., 2020) | matteoiacoviello.com/tpu.htm | Monthly | As published |
| GPR | Geopolitical Risk index (Caldara & Iacoviello, 2022) | matteoiacoviello.com/gpr.htm | Monthly | As published |
| MPU | US Monetary Policy Uncertainty index (Baker, Bloom & Davis) | policyuncertainty.com | Monthly | As published |
| US_EPU | US Economic Policy Uncertainty index, news-based (Baker, Bloom & Davis) | policyuncertainty.com | Monthly | As published |
| US_VIX | CBOE Volatility Index, daily close (FRED series VIXCLS) | FRED, Federal Reserve Bank of St. Louis | Daily | Calendar-month mean |

**Harmonisation.** Every series is dated to the first day of its month. Daily series are averaged over the calendar month; monthly series are only re-dated. The series are merged on the months common to all of them, which gives a balanced panel from April 1999 to December 2025 (321 months) with no missing values. All harmonised series are compiled into a single file, `data/Compiled.xlsx` (sheet `X_var`), so that every model in this notebook loads the same data from one place.

**Data availability.** The CEIC series (IIP, CPI, exchange rate, call money rate) are proprietary and available only by subscription, so they are not included in the public repository; `data/README.md` lists the series and where to place them. The other series are publicly available.

**Cleaning.** All series were screened for observations more than four standard deviations from their mean (Section 2 below). One data error was found and corrected: the call money rate for March 2007 is recorded as 54.32%, whereas the RBI *Handbook of Statistics on the Indian Economy* reports a monthly weighted average of 14.07% (the 80% figure that month was an intraday high). The remaining extreme values are genuine episodes, such as the April 2020 collapse in industrial output, the September 2001 and March 2003 peaks in geopolitical risk, and the 2025 surge in trade policy uncertainty.

**Constructed variables.** The output gap is built from the three IIP base-year series by ratio splicing and Hodrick–Prescott filtering (Section 3 below). Monthly log returns, the 12-month rolling volatility and the TPU × GPR interaction are computed from the merged data in the analysis notebook.

## 2. Screening and cleaning of the compiled panel

The compiled panel (`Compiled.xlsx`, sheet `X_var`) is checked for completeness and screened for observations more than four standard deviations from their mean. Each flagged value is then checked against the original source. One data error is corrected: the call money rate for March 2007 is recorded as 54.32%, whereas the RBI *Handbook of Statistics on the Indian Economy* reports a monthly weighted average of 14.07%.

In [ ]:
BASE_VARS = ["TPU", "GPR", "FX_USD", "CPI_GR", "Int_Rate", "Output_Gap", "MPU", "US_EPU", "US_VIX"]
macro = pd.read_excel(BASE / "data" / "Compiled.xlsx", sheet_name="X_var")
macro["Date"] = pd.to_datetime(macro["Date"])
macro = macro.set_index("Date").sort_index()[BASE_VARS]

full_months = pd.date_range(macro.index.min(), macro.index.max(), freq="MS")
print(f"Panel: {macro.index.min():%b-%Y} – {macro.index.max():%b-%Y}, {len(macro)} months; "
      f"months missing from calendar: {len(full_months.difference(macro.index))}; missing values: {int(macro.isna().sum().sum())}")

z = (macro - macro.mean()) / macro.std()
flag = z.abs().stack()
flag = flag[flag > 4].rename("|z|").reset_index().rename(columns={"level_1": "Variable"})
flag["Value"] = [macro.loc[d, v] for d, v in zip(flag["Date"], flag["Variable"])]
flag["Date"] = flag["Date"].dt.strftime("%b-%Y")
save_table(flag.round(2), "T01_outlier_screen", index=False)

# correction of the March-2007 call money rate (RBI Handbook monthly weighted average)
print("Int_Rate Mar-2007 as compiled:", macro.loc["2007-03-01", "Int_Rate"])
macro.loc["2007-03-01", "Int_Rate"] = 14.07
print("Int_Rate Mar-2007 corrected  :", macro.loc["2007-03-01", "Int_Rate"])

## 3. Construction of the output gap
India's Index of Industrial Production (IIP) is published by MOSPI on three successive base years: 1993-94, 2004-05 and
2011-12. Each base uses a different basket of items and weights, so index levels on different bases are not comparable.
This appendix builds a single monthly IIP series on the 2011-12 base by ratio splicing, extracts the cyclical component
with the Hodrick–Prescott filter, and checks that the result reproduces the `Output_Gap` series used in the analysis.

Input files (CEIC exports, sheet `My Series`, column B = series on its own base):

| File | Base | Coverage in file |
|---|---|---|
| `data/IIP/IPI_94.xlsx` | 1993-94 = 100 | Jan-1996 – Jun-2011 |
| `data/IIP/IPI_5.xlsx`  | 2004-05 = 100 | Apr-2005 – Feb-2017 |
| `data/IIP/IPI_11.xlsx` | 2011-12 = 100 | Apr-2012 – Dec-2025 |

In [ ]:
from statsmodels.tsa.filters.hp_filter import hpfilter

HP_LAMBDA = 14_400                                   # standard smoothing parameter for monthly data

def load_ceic_series(path, col=1, sheet="My Series"):
    """Read one column of a CEIC export: metadata rows on top, then Date | value rows."""
    raw = pd.read_excel(path, sheet_name=sheet, header=None)
    dates = pd.to_datetime(raw[0], errors="coerce")
    s = pd.Series(pd.to_numeric(raw[col], errors="coerce").values, index=dates)
    s = s[s.index.notna()].dropna().sort_index()
    s.index = s.index.to_period("M").to_timestamp()
    return s

iip94 = load_ceic_series(IIP_DIR / "IPI_94.xlsx")
iip04 = load_ceic_series(IIP_DIR / "IPI_5.xlsx")
iip11 = load_ceic_series(IIP_DIR / "IPI_11.xlsx")

cover = pd.DataFrame({
    "Base": ["1993-94", "2004-05", "2011-12"],
    "First month": [s.index.min().strftime("%b-%Y") for s in (iip94, iip04, iip11)],
    "Last month": [s.index.max().strftime("%b-%Y") for s in (iip94, iip04, iip11)],
    "Observations": [len(s) for s in (iip94, iip04, iip11)]})
cover["Overlap with next base (months)"] = [len(iip94.index.intersection(iip04.index)),
                                            len(iip04.index.intersection(iip11.index)), np.nan]
save_table(cover, "TA1_iip_base_coverage", index=False)

### 3.1 Ratio splicing
Each older series is rescaled to the newer base using the ratio of the two indices in the **first month of the newer
base**, when both are published:

$$\widetilde{IIP}^{\,04}_t = IIP^{04}_t\times\frac{IIP^{11}_{\text{Apr 2012}}}{IIP^{04}_{\text{Apr 2012}}},\qquad
  \widetilde{IIP}^{\,94}_t = IIP^{94}_t\times\frac{\widetilde{IIP}^{\,04}_{\text{Apr 2005}}}{IIP^{94}_{\text{Apr 2005}}}$$

The spliced series takes the rescaled 1993-94 values up to Mar-2005, the rescaled 2004-05 values from Apr-2005 to
Mar-2012, and the published 2011-12 values from Apr-2012. Month-on-month growth rates within each base are unchanged.

In [ ]:
def ratio_splice(old, new, link):
    """Rescale `old` to the units of `new` using their ratio at month `link`."""
    link = pd.Timestamp(link)
    factor = new.loc[link] / old.loc[link]
    return old * factor, factor

LINK_04_11, LINK_94_04 = "2012-04-01", "2005-04-01"
iip04_l, f_04_11 = ratio_splice(iip04, iip11, LINK_04_11)
iip94_l, f_94_04 = ratio_splice(iip94, iip04_l, LINK_94_04)

iip = pd.concat([iip94_l.loc[:pd.Timestamp(LINK_94_04) - pd.offsets.MonthBegin(1)],
                 iip04_l.loc[LINK_94_04:pd.Timestamp(LINK_04_11) - pd.offsets.MonthBegin(1)],
                 iip11.loc[LINK_04_11:]]).rename("IIP_2011_12")
assert iip.index.is_unique
assert len(pd.date_range(iip.index.min(), iip.index.max(), freq="MS").difference(iip.index)) == 0, "gap in spliced IIP"

save_table(pd.DataFrame({"Link": ["2004-05 → 2011-12", "1993-94 → 2004-05 (linked)"],
                         "Link month": ["Apr-2012", "Apr-2005"], "Ratio factor": [f_04_11, f_94_04]}).round(5),
           "TA2_iip_splice_factors", index=False)

# Cross-check against the spliced series supplied by CEIC (sheet `My Series (2)`)
ceic_spliced = load_ceic_series(IIP_DIR / "IPI_11.xlsx", col=2)
print(f"Spliced IIP: {iip.index.min():%b-%Y} – {iip.index.max():%b-%Y}, N = {len(iip)}; "
      f"max |difference| from CEIC spliced series = {(iip - ceic_spliced).abs().max():.2e}")

### 3.2 Hodrick–Prescott output gap
With $y_t=\ln IIP_t$, the HP trend $\tau_t$ solves
$\min_{\tau}\sum_t (y_t-\tau_t)^2+\lambda\sum_t[(\tau_{t+1}-\tau_t)-(\tau_t-\tau_{t-1})]^2$ with $\lambda=14{,}400$.
The output gap is $100\,(y_t-\tau_t)$, the percentage deviation of industrial output from trend. The filter is run on the
full spliced sample (Jan-1996 – Dec-2025) so that the 1999–2025 estimation window is not affected by the end-point
problem at the start of the series.

In [ ]:
cycle, trend = hpfilter(np.log(iip), lamb=HP_LAMBDA)
gap = (100 * cycle).rename("Output_Gap_rebuilt")
iip_trend = np.exp(trend)

check = pd.concat([gap, macro["Output_Gap"]], axis=1).dropna()
print(f"Rebuilt vs compiled Output_Gap ({check.index.min():%b-%Y} – {check.index.max():%b-%Y}): "
      f"max |difference| = {(check.iloc[:, 0] - check.iloc[:, 1]).abs().max():.2e}")

# Sensitivity of the gap to the splicing rule: link on the average ratio over the first 12 overlapping months
def avg_ratio_splice(old, new, start):
    win = pd.date_range(start, periods=12, freq="MS")
    factor = (new.loc[win] / old.loc[win]).mean()
    return old * factor
iip04_a = avg_ratio_splice(iip04, iip11, LINK_04_11)
iip94_a = avg_ratio_splice(iip94, iip04_a, LINK_94_04)
iip_alt = pd.concat([iip94_a.loc[:"2005-03-01"], iip04_a.loc["2005-04-01":"2012-03-01"], iip11.loc["2012-04-01":]])
gap_alt = 100 * hpfilter(np.log(iip_alt), lamb=HP_LAMBDA)[0]
sens = pd.DataFrame({"Splicing rule": ["Single-month ratio (baseline)", "12-month average ratio"],
                     "Gap SD (%)": [gap.std(), gap_alt.std()],
                     "Corr. with baseline": [1.0, gap.corr(gap_alt)],
                     "Max |diff| from baseline (pp)": [0.0, (gap - gap_alt).abs().max()]})
save_table(sens.round(4), "TA3_output_gap_splice_sensitivity", index=False)

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 9.5), sharex=True)
for s, lab, col in [(iip94, "1993-94 = 100", C["blue"]), (iip04, "2004-05 = 100", C["orange"]), (iip11, "2011-12 = 100", C["aqua"])]:
    ax[0].plot(s.index, s, color=col, lw=1.2, label=lab)
ax[0].set_yscale("log"); ax[0].set_ylabel("Index (log scale)"); ax[0].legend(loc="upper left", title="Published base")
ax[0].set_title("IIP as published on three base years")
ax[1].plot(iip.index, iip, color=C["blue"], lw=1.1, label="Spliced IIP (2011-12 = 100)")
ax[1].plot(iip.index, iip_trend, color=C["orange"], lw=1.6, label=f"HP trend (λ = {HP_LAMBDA:,})")
for d in (LINK_94_04, LINK_04_11): ax[1].axvline(pd.Timestamp(d), color=C["muted"], ls="--", lw=0.8)
ax[1].set_ylabel("Index"); ax[1].legend(loc="upper left"); ax[1].set_title("Spliced IIP and HP trend (dashed lines: link months)")
ax[2].bar(gap.index, gap, width=25, color=np.where(gap >= 0, C["blue"], C["orange"]))
ax[2].axhline(0, color=C["axis"], lw=0.8); ax[2].set_ylabel("% of trend"); ax[2].set_title("Output gap")
year_axis(ax[2]); fig.tight_layout(); save_fig(fig, "FA1_iip_splicing_output_gap")

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(gap.index, gap, color=C["blue"], lw=1.1, label="Output gap")
ax.set_ylim(-20, 15); ax.axhline(0, color=C["axis"], lw=0.8)
ax.annotate(f"Apr-2020: {gap.loc['2020-04-01']:.1f}%", xy=(pd.Timestamp("2020-04-01"), -20), xytext=(pd.Timestamp("2014-01-01"), -17),
            arrowprops=dict(arrowstyle="->", color=C["ink2"]), fontsize=9, color=C["ink2"])
ax.set_ylabel("% of trend"); ax.set_title("Output gap, truncated scale (COVID observation off-scale)"); year_axis(ax)
fig.tight_layout(); save_fig(fig, "FA2_output_gap_zoom")

## 4. Output

The rebuilt output gap is identical to the `Output_Gap` column of `Compiled.xlsx` (the check above prints the maximum difference). The cell below saves the spliced IIP, its HP trend and the output gap so they can be inspected or reused.

In [ ]:
out = pd.DataFrame({"IIP_spliced_2011_12": iip, "IIP_HP_trend": iip_trend, "Output_Gap": gap})
out.index.name = "Date"
out.to_excel(BASE / "outputs" / "output_gap_construction.xlsx")
print("Saved outputs/output_gap_construction.xlsx", out.shape)
out.tail()